In [2]:
import pandas as pd

In [3]:
shark_attack = pd.read_csv("./shark_clean_final.csv")

In [4]:
shark_attack.head(10)

,Country,State,Location,Activity,Date,Year,Type,Month,Sex,Age,Injury_Category,Time,Species,Fatal Y/N
0,Australia,Western Australia,Sorrento Beach Perth,Swimming,18 september,2026,Unprovoked,September,M,63.0,Other,10:15,White Shark,Y
1,Canada,Quebec,Off The Coast Of Perce Le Bilbo Dive Site,Diving,16 september,2026,Unprovoked,September,M,NaN,Injury,10:30,White Shark,N
2,Bahamas,Bimini,Bimini Island,Swimming,14 september,2026,Unprovoked,September,F,37.0,Injury,17:30,Other/Unknown,N
3,Australia,Western Australia,Geraldton,Surfing,13 september,2026,Unprovoked,September,M,NaN,Severe injury,09:45,Other/Unknown,N
4,Usa,Hawaii,Honolulu,Surfing,7 september,2026,Unprovoked,September,M,24.0,Other,16:40,Tiger Shark,N
5,Usa,Florida,New Smyrna Beach,Surfing,2 september,2026,Unprovoked,September,M,NaN,Injury,11:00,Other/Unknown,N
6,Usa,Florida,30 Miles Off Pensacola,Spearfishing,1 september,2026,Unprovoked,September,M,40.0,Injury,16:35,Bull Shark,N
7,Usa,Massachusetts,Norton Point Edgartown Marthas Vinyard,Swimming,31 august,2026,Unprovoked,August,F,NaN,Injury,13:40,Other/Unknown,N
8,Usa,Hawaii,"Ala Moana Beach Park, Oahu",Surfing,26 august,2026,Unprovoked,August,F,NaN,No injury,13:30,Tiger Shark,N
9,Usa,Florida,Boca Grande (Island In The Keys),Wading on a sandbar,23 august,2026,Unprovoked,August,M,43.0,Injury,17:00,Other/Unknown,N


## Checking statistics of fatalities

In [5]:
# Calculating percentage of fatal attacks recorded

sum_fatal = (shark_attack['Fatal Y/N']).count()
yes_fatal = (shark_attack['Fatal Y/N'] == 'Y').sum()

ratio_fatal = (yes_fatal / sum_fatal) * 100
print(ratio_fatal.round(2)) # only 11% of all attacks recorded were fatal

11.18


In [6]:
# Finding the most lethal year

fatal_by_year = shark_attack[shark_attack['Fatal Y/N'] == 'Y']['Year'].value_counts()
deadliest_year = fatal_by_year.idxmax()
deadliest_count = fatal_by_year.max()

print(f"The most lethal year was {int(deadliest_year)} with {deadliest_count} fatal attacks.")

The most lethal year was 2023 with 19 fatal attacks.


## Filtering the data to the target time period - last 10 years.

In [7]:
# Create a dedicated frame for the 10-year period
recent_sharks = shark_attack[shark_attack['Year'].between(2016, 2026)].copy()

In [8]:
# Total attacks per over the 10 years
attack_per_year = recent_sharks['Year'].value_counts().sort_index()

In [9]:
#  Filter down to Provoked and Unprovoked attacks
type_subset = recent_sharks[recent_sharks['Type'].isin(['Provoked', 'Unprovoked'])]

#  Compare them year by year
type_comparison = pd.crosstab(index=type_subset['Year'], columns=type_subset['Type'])

In [10]:
# Filter for only Unprovoked
unprovoked_attacks = recent_sharks[recent_sharks['Type'] == 'Unprovoked']

# "unprovoked" attack count per year
unprovoked_per_year = unprovoked_attacks['Year'].value_counts().sort_index()

In [11]:
#  Filter down Fatal to Yes or No
fatal_subset = recent_sharks[recent_sharks['Fatal Y/N'].isin(['Y', 'N'])]

#  Compare them year by year
fatal_comparison = pd.crosstab(index=fatal_subset['Year'], columns=fatal_subset['Fatal Y/N'])

In [12]:
# Filter for Fatal "Y" only
fatal_attacks = recent_sharks[recent_sharks['Fatal Y/N'] == 'Y']

# fatal attack count per year
fatal_per_year = fatal_attacks['Year'].value_counts().sort_index()

In [13]:
# summarise all data into a separate data frame
data_summary = pd.DataFrame({
    "Total attacks": attack_per_year,
    "Unprovoked attacks": unprovoked_per_year,
    "Provoked": type_comparison["Provoked"],
    "Fatal attacks": fatal_per_year,
    "Non-Fatal attacks": fatal_comparison["N"]
}).fillna(0) # fills 0 if a year has zeron incidents in any category

data_summary

,Total attacks,Unprovoked attacks,Provoked,Fatal attacks,Non-Fatal attacks
Year,,,,,
2016,133,105,12,8,114
2017,141,109,7,8,120
2018,124,92,13,6,115
2019,114,90,11,10,99
2020,101,86,5,14,85
2021,111,94,11,13,95
2022,98,81,12,12,85
2023,109,86,9,19,85
2024,52,49,2,9,43


## Calculating the percentage change over those years

In [14]:
start_year = data_summary.iloc[0] #2026
end_year = data_summary.iloc[-1] #2016

# the standard formula to calculate percentage increase/decrease is: 
# percentage change = ((recent value/old value)-1)*100

pct_total = (end_year["Total attacks"] / start_year["Total attacks"] - 1) * 100
pct_unprovoked = (end_year["Unprovoked attacks"] / start_year["Unprovoked attacks"] - 1) * 100
pct_fatal = (end_year["Fatal attacks"] / start_year["Fatal attacks"] - 1) * 100

print(f"Changes from {start_year.name} to {end_year.name}:")
print(f"Total Attacks: {pct_total:+.1f}%")
print(f"Unprovoked Attacks: {pct_unprovoked:+.1f}%")
print(f"Fatal Attacks: {pct_fatal:+.1f}%")

Changes from 2016 to 2026:
Total Attacks: -54.1%
Unprovoked Attacks: -48.6%
Fatal Attacks: +12.5%


In [15]:
# Calculate Fatality Rate as a percentage of total attacks each year
data_summary["Fatality_Rate"] = (data_summary["Fatal attacks"] / data_summary["Total attacks"]) * 100

# Compare start rate vs end rate
start_rate = data_summary["Fatality_Rate"].iloc[0]
end_rate = data_summary["Fatality_Rate"].iloc[-1]
difference = end_rate - start_rate

print(f"Fatality Rate in {start_year.name}: {start_rate:.1f}%")
print(f"Fatality Rate in {end_year.name}: {end_rate:.1f}%")
print(f"Change: {difference:+.1f}%")

Fatality Rate in 2016: 6.0%
Fatality Rate in 2026: 14.8%
Change: +8.7%


## Short and Extensive Summary of the Outcomes